# Smart MCQ Solver — Milestone 5
**Email:** 23f3001763@ds.study.iitm.ac.in

This notebook covers the 10 questions in Milestone 5:
- Loading Sequence Classification models (DeBERTa, RoBERTa)
- Simple vs Weighted Ensembling
- Test-Time Augmentation (TTA)
- Confidence Gain Analysis
- Computing final MAP@3 score on validation

> **Important:** Ensure that the notebook has **Internet Access** enabled in Kaggle settings so it can download the pretrained models.
> Make sure the competition data (`train.csv`, `test.csv`) is added to the notebook's input.

In [ ]:
# Install necessary libraries
!pip install -q transformers datasets torch pandas scikit-learn sentencepiece

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Set Global Seeds for Reproducibility ──────────────────────────────────────
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ── Load datasets ─────────────────────────────────────────────────────────────
test_path = '/kaggle/input/smart-mcq-solver-challenge/test.csv'
train_path = '/kaggle/input/smart-mcq-solver-challenge/train.csv'

if not os.path.exists(test_path):
    test_path = 'test.csv' # local fallback
if not os.path.exists(train_path):
    train_path = 'train.csv'

# If test.csv doesn't exist at all (e.g. testing locally), mock it with train.csv
if os.path.exists(test_path):
    test_df = pd.read_csv(test_path)
else:
    print("Warning: test.csv not found, using train.csv as mock test data.")
    test_df = pd.read_csv(train_path)

train_df = pd.read_csv(train_path)
print("Datasets loaded successfully!")

In [ ]:
# ── Define Inference Helper ───────────────────────────────────────────────────
def get_probs(df, model, tokenizer, batch_size=16, prepend_text=""):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    all_probs = []
    
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        texts = []
        for _, row in batch.iterrows():
            # Format: prompt [SEP] A [SEP] B [SEP] C [SEP] D [SEP] E
            text = prepend_text + str(row['prompt']) + " " + tokenizer.sep_token + " " + \
                   str(row.get('A', '')) + " " + tokenizer.sep_token + " " + \
                   str(row.get('B', '')) + " " + tokenizer.sep_token + " " + \
                   str(row.get('C', '')) + " " + tokenizer.sep_token + " " + \
                   str(row.get('D', '')) + " " + tokenizer.sep_token + " " + \
                   str(row.get('E', ''))
            texts.append(text)
            
        inputs = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.extend(probs)
            
    return np.array(all_probs)

---
## Load Models & Generate Base Probabilities

In [ ]:
# Load Models
import warnings
warnings.filterwarnings("ignore") # Ignore missing weights warnings since we are dynamically loading a classification head

deb_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")
deb_model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-small", num_labels=5, ignore_mismatched_sizes=True)

rob_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
rob_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5, ignore_mismatched_sizes=True)

labels = ['A', 'B', 'C', 'D', 'E']

# Get probabilities for row 25
row_25 = test_df.iloc[[25]]
deb_probs_25 = get_probs(row_25, deb_model, deb_tokenizer)[0]
rob_probs_25 = get_probs(row_25, rob_model, rob_tokenizer)[0]

---
## Question 1
Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

In [ ]:
deb_max_idx = np.argmax(deb_probs_25)
q1_ans_label = labels[deb_max_idx]
q1_ans_prob = deb_probs_25[deb_max_idx]
print(f">>> Q1 Answer: {q1_ans_label}, {q1_ans_prob:.4f}")

---
## Question 2
Which answer option receives the highest averaged probability after simple probability ensembling?

In [ ]:
avg_probs_25 = (deb_probs_25 + rob_probs_25) / 2
avg_max_idx = np.argmax(avg_probs_25)
q2_ans_label = labels[avg_max_idx]
print(f">>> Q2 Answer: {q2_ans_label}")

---
## Question 3
Which answer option is ranked first after weighted ensembling (DeBERTa 0.70, RoBERTa 0.30)?

In [ ]:
weight_probs_25 = (0.7 * deb_probs_25) + (0.3 * rob_probs_25)
weight_max_idx = np.argmax(weight_probs_25)
q3_ans_label = labels[weight_max_idx]
print(f">>> Q3 Answer: {q3_ans_label}")

---
## Question 4
What is the Top-3 prediction string for row index 25?

In [ ]:
top3_idx_25 = np.argsort(weight_probs_25)[::-1][:3]
q4_ans = " ".join([labels[i] for i in top3_idx_25])
print(f">>> Q4 Answer: {q4_ans}")

---
## Question 5
Run weighted ensemble pipeline on every row of test.csv. Save submission.csv. How many prediction rows in file (excluding header)?

In [ ]:
print("Running full dataset inference (this might take a moment)...")
deb_probs_all = get_probs(test_df, deb_model, deb_tokenizer)
rob_probs_all = get_probs(test_df, rob_model, rob_tokenizer)
weight_probs_all = (0.7 * deb_probs_all) + (0.3 * rob_probs_all)

preds = []
for probs in weight_probs_all:
    top3_idx = np.argsort(probs)[::-1][:3]
    preds.append(" ".join([labels[i] for i in top3_idx]))
    
submission = pd.DataFrame({'id': test_df['id'] if 'id' in test_df else range(len(test_df)), 'prediction': preds})
submission.to_csv('submission.csv', index=False)
q5_ans = len(submission)
print(f">>> Q5 Answer: {q5_ans}")

---
## Question 6
First 50 rows of test.csv. Test-Time Augmentation (TTA). How many produce a different Top-1 prediction?

In [ ]:
test_50 = test_df.iloc[:50]
deb_probs_50_orig = deb_probs_all[:50] # reuse calculations for speed
deb_probs_50_tta = get_probs(test_50, deb_model, deb_tokenizer, prepend_text="Answer the following multiple-choice question carefully: ")

deb_tta_avg = (deb_probs_50_orig + deb_probs_50_tta) / 2

orig_top1 = np.argmax(deb_probs_50_orig, axis=1)
tta_top1 = np.argmax(deb_tta_avg, axis=1)

q6_ans = np.sum(orig_top1 != tta_top1)
print(f">>> Q6 Answer: {q6_ans}")

---
## Question 7
First 100 rows of test.csv. DeBERTa vs Weighted Ensemble Top-1 differences.

In [ ]:
deb_probs_100 = deb_probs_all[:100]
weight_probs_100 = weight_probs_all[:100]

deb_top1_100 = np.argmax(deb_probs_100, axis=1)
weight_top1_100 = np.argmax(weight_probs_100, axis=1)

q7_ans = np.sum(deb_top1_100 != weight_top1_100)
print(f">>> Q7 Answer: {q7_ans}")

---
## Question 8
First 100 rows. How many rows have a positive confidence gain (Ensemble - DeBERTa > 0)?

In [ ]:
deb_conf = np.max(deb_probs_100, axis=1)
weight_conf = np.max(weight_probs_100, axis=1)
gain = weight_conf - deb_conf
q8_ans = np.sum(gain > 0)
print(f">>> Q8 Answer: {q8_ans}")

---
## Question 9
First 100 rows. Compare Top-3 prediction strings generated by DeBERTa alone vs Weighted Ensemble. How many rows have at least one change?

In [ ]:
deb_top3_idx = np.argsort(deb_probs_100, axis=1)[:, ::-1][:, :3]
weight_top3_idx = np.argsort(weight_probs_100, axis=1)[:, ::-1][:, :3]

diff_count = 0
for d_idx, w_idx in zip(deb_top3_idx, weight_top3_idx):
    if not np.array_equal(d_idx, w_idx):
        diff_count += 1

q9_ans = diff_count
print(f">>> Q9 Answer: {q9_ans}")

---
## Question 10
Using Weighted Ensemble on first 100 validation samples, compute MAP@3.

In [ ]:
train_100 = train_df.iloc[:100]
deb_train_100 = get_probs(train_100, deb_model, deb_tokenizer)
rob_train_100 = get_probs(train_100, rob_model, rob_tokenizer)
weight_train_100 = (0.7 * deb_train_100) + (0.3 * rob_train_100)

weight_train_top3 = np.argsort(weight_train_100, axis=1)[:, ::-1][:, :3]
train_preds = [[labels[i] for i in row] for row in weight_train_top3]
train_actuals = train_100['answer'].tolist()

def mapk(actuals, predictions, k=3):
    scores = []
    for a, p in zip(actuals, predictions):
        score = 0.0
        for rank, pred in enumerate(p[:k], start=1):
            if pred == a:
                score = 1.0 / rank
                break
        scores.append(score)
    return np.mean(scores)

q10_ans = mapk(train_actuals, train_preds)
print(f">>> Q10 Answer: {q10_ans:.4f}")